In [27]:
import os
import numpy as np
import pandas as pd
import torch

In [28]:
import re

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)   # Remove HTML tags
    text = re.sub(r"\r\n|\r|\n", " ", text)  # Remove line breaks
    text = re.sub(r"\s+", " ", text)     # Normalize whitespace
    text = text.strip()

    return text

In [33]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_path = "../models/saved_summary_model/"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

model.generation_config.max_length = None

model.to(device)
model.eval()

print("device:", device)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

device: cpu


In [34]:
def summarize_dialogue(dialogue):
    dialogue = "summarize: " + clean_data(dialogue)

    # tokenization (no padding needed for a single example)
    inputs = tokenizer(
        dialogue,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate summary :- token id's will be generated
    with torch.no_grad():
        targets = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    # token id's to summary
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)

    return summary

In [35]:
def summarize_batch(dialogues, batch_size=16, **overrides):
    gen_kwargs = dict(
        max_new_tokens=128,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    gen_kwargs.update(overrides)

    texts = ["summarize: " + clean_data(d) for d in dialogues]

    # sort by length so each batch has little padding, then restore the original order
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    summaries = [None] * len(texts)

    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]

        inputs = tokenizer(
            [texts[i] for i in idx],
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, s in zip(idx, decoded):
            summaries[i] = s

    return summaries

### Testing on Dialogue

In [36]:
test_dialogue = """
Sarah: Hey John, did you finish the project report?
John: Almost. I completed the analysis section yesterday.
Sarah: Great. What about the introduction and conclusion?
John: I finished the introduction this morning, but I still need to write the conclusion.
Sarah: When do you think you'll finish it?
John: I should be able to complete it by tonight and send the final report tomorrow morning.
Sarah: Perfect. I'll review it once you send it.
"""

In [37]:
summary = summarize_dialogue(test_dialogue)

print("Summary is :- ", summary)

Summary is :-  John finished the analysis section yesterday. He needs to write the conclusion. He should be able to complete it by tonight and send the final report tomorrow morning. Sarah will review it once he sends it.


### Test on Multiple Unseen Dialogues

In [38]:
test_dialogue_1 = """
Manager: We have a team meeting scheduled for Friday at 10 AM.
Alex: I won't be available at that time because I have a doctor's appointment.
Manager: That's fine. Can you join the meeting at 2 PM instead?
Alex: Yes, 2 PM works for me.
Manager: Great. I'll move the meeting to Friday at 2 PM.
"""

test_dialogue_2 = """
Emma: We need to buy some groceries for dinner.
Tom: What should we get?
Emma: We need vegetables, rice, and some chicken.
Tom: I'll buy the vegetables and rice.
Emma: Great. I'll pick up the chicken on my way home.
Tom: Perfect. Let's meet at home around 7 PM.
"""

test_dialogue_3 = """
Rahul: Have you completed the machine learning project?
Amit: I finished the model training yesterday.
Rahul: What about the documentation?
Amit: I'm working on it now and should finish it by tomorrow.
Rahul: Okay. I'll prepare the presentation after you finish the documentation.
Amit: Sounds good. I'll send it to you tomorrow evening.
"""

In [39]:
summary_1 = summarize_dialogue(test_dialogue_1)
summary_2 = summarize_dialogue(test_dialogue_2)
summary_3 = summarize_dialogue(test_dialogue_3)

print("SUMMARY 1:")
print(summary_1)

print("\nSUMMARY 2:")
print(summary_2)

print("\nSUMMARY 3:")
print(summary_3)

SUMMARY 1:
Manager has a team meeting scheduled for Friday at 10 AM. Alex will join the meeting at 2 PM instead.

SUMMARY 2:
Emma and Tom are going to buy vegetables, rice and some chicken for dinner. They will meet at home around 7 PM.

SUMMARY 3:
Amit finished the model training yesterday and should finish the documentation by tomorrow. Rahul will prepare the presentation after Amit finishes the documentation.


### ROUGE Evaluation

In [40]:
from rouge_score import rouge_scorer

In [43]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

model.eval()

for i in range(200):

    dialogue = test_dataset[i]["dialogue"]
    reference = test_dataset[i]["summary"]

    generated = summarize_dialogue(dialogue)

    scores = scorer.score(reference, generated)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("Average ROUGE-1:", sum(rouge1_scores) / len(rouge1_scores))
print("Average ROUGE-2:", sum(rouge2_scores) / len(rouge2_scores))
print("Average ROUGE-L:", sum(rougeL_scores) / len(rougeL_scores))

Average ROUGE-1: 0.49893685359819606
Average ROUGE-2: 0.24022834135337934
Average ROUGE-L: 0.408611483802744


In [45]:
for i in range(5):

    dialogue = test_dataset[i]["dialogue"]
    reference = test_dataset[i]["summary"]
    generated = summarize_dialogue(dialogue)

    print(f"\n{'=' * 80}")
    print(f"Example {i + 1}")
    
    print("\nDIALOGUE:")
    print(dialogue)

    print("\nREFERENCE SUMMARY:")
    print(reference)

    print("\nGENERATED SUMMARY:")
    print(generated)


Example 1

DIALOGUE:
Hannah: Hey, do you have Betty's number? Amanda: Lemme check Hannah: Amanda: Sorry, can't find it. Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: I don't know him well Hannah: Amanda: Don't be shy, he's very nice Hannah: If you say so.. Hannah: I'd rather you texted him Amanda: Just text him 🙂 Hannah: Urgh.. Alright Hannah: Bye Amanda: Bye bye

REFERENCE SUMMARY:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

GENERATED SUMMARY:
Amanda can't find Betty's number. Larry called her last time they were at the park together.

Example 2

DIALOGUE:
Eric: MACHINE! Rob: That's so gr8! Eric: I know! And shows how Americans see Russian ;) Rob: And it's really funny! Eric: I know! I especially like the train part! Rob: Hahaha! No one talks to the machine like that! Eric: Is this his only stand-up? Rob: Idk. I'll check. Eric: Sure. Rob: Turns out no! There are some of his stand-ups on youtube. Eric: G